In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers


train_df = pd.read_csv('sign_mnist_train.csv')
test_df = pd.read_csv('sign_mnist_test.csv')

y_train = train_df['label'].values
X_train = train_df.drop('label', axis=1).values
y_test = test_df['label'].values
X_test = test_df.drop('label', axis=1).values

X_train = X_train.reshape(-1,28,28,1) / 255.0
X_test = X_test.reshape(-1,28,28,1) /255.0

anomaly_class=0

X_train_normal = X_train[y_train!=anomaly_class]


X_test_normal = X_test[y_test != anomaly_class]
X_test_anomalies = X_test[y_test == anomaly_class]

model = keras.Sequential([
    layers.Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2), padding='same'),
    layers.Conv2D(8, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2), padding='same'),

    layers.Conv2D(8,(3,3), activation='relu', padding='same'),
    layers.UpSampling2D((2,2)),
    layers.Conv2D(16,(3,3), activation='relu', padding='same'),
    layers.UpSampling2D((2,2)),
    layers.Conv2D(1,(3,3), activation='sigmoid',padding='same')
])


model.compile(optimizer='adam',loss='mse',metrics=['mae'])

H = model.fit(X_train_normal, X_train_normal, epochs=10, batch_size=32, validation_split=0.2)

X_test_normal_preds = model.predict(X_test_normal)
normal_mse = np.mean(np.power(X_test_normal-X_test_normal_preds,2),axis=(1,2,3))
threshold = np.percentile(normal_mse,99)

anomaly_preds = model.predict(X_test_anomalies)
anomaly_mse = np.mean(np.power(X_test_anomalies - anomaly_preds,2), axis=(1,2,3))

detected_anomalies = anomaly_mse > threshold

print(f"Detected Anomalies out of {len(X_test_anomalies)} actual anomaly images: {sum(detected_anomalies)}") 
